- https://medium.com/data-science/building-knowledge-graphs-with-llm-graph-transformer-a91045c49b59 (passing the list of available relations and entities in the prompt, limited number of relation types)
- https://medium.com/data-science/building-knowledge-graphs-with-llm-graph-transformer-a91045c49b59  (can the llm be prompted to add, if needed, extra relation and node types to the restricted list - agent)
- https://bratanic-tomaz.medium.com/constructing-knowledge-graphs-from-text-using-openai-functions-096a6d010c17 (Entity deduplication / disambiguation)

In [ ]:
import os
from langchain_core.documents import Document
from langchain_ollama import ChatOllama


LLM_MODEL_NAME="jaro-jm/llm:latest"
# LLM_MODEL_NAME="SpeakLeash/bielik-11b-v2.3-instruct:Q8_0"
os.environ["HSA_OVERRIDE_GFX_VERSION"]="11.0.0"

llm_text = ChatOllama(model=LLM_MODEL_NAME, temperature=0, reasoning=False, num_ctx=25000, num_predict=15000)

text = """Donald Trump od początku traktował konflikt w Ukrainie jako nieswoją wojnę. Nie miał cierpliwości ani zrozumienia dla ukraińskich racji. Błędnie zakładał też, że zdoła oderwać Rosję od sojuszu z Chinami. Tymczasem oba kraje mocno zacieśniły współpracę gospodarczą, a Chiny dostarczają dziś Rosjanom maszyny i technologie, na które monopol miał kiedyś Zachód. Moskwa i Pekin pamiętają czasy, gdy były podzielone. Putin i Xi są zgodni, że rozpad Związku Radzieckiego nie był koniecznością, tylko skutkiem błędów Kremla. Rosję i Chiny dzieli przepaść kulturalna, bo Rosjanie nie chcą być Chińczykami, ale to akurat ułatwia relacje. Współpraca z Europą grozi w oczach Putina zarażeniem rosyjskiego społeczeństwa „demokratycznym wirusem”, a w kontaktach z Chinami takiego ryzyka nie ma. Rosja może jednak wpaść w nieodwracalną zależność od Chin. Decydując się na inwazję na Ukrainę, Putin chciał przede wszystkim zasilić ludność Rosji dziesiątkami milionów Słowian, co było dla niego ważniejsze od zdobyczy terytorialnych. W tle są obawy przed chińską dominacją."""

In [ ]:
from langchain_experimental.graph_transformers import LLMGraphTransformer

allowed_nodes = [
    "Person",
    "Country",
    "Organization",
    "Alliance",
    "Conflict",
    "Event",
    "Ideology",
    "Technology",
    "EconomicSector",
    "Culture",
    "HistoricalEvent",
    "Policy"
]
allowed_relationships = [
    "PERCEIVES_AS",
    "ASSUMES_POSSIBILITY_OF",
    "STRENGTHENS_COOPERATION",
    "SUPPLIES_TECHNOLOGY",
    "DRAWS_HISTORICAL_LESSONS",
    "DENIES_NECESSITY_OF",
    "DIVIDES",
    "FEARS_DEMOCRATIZATION",
    "BECOMES_DEPENDENT_ON",
    "AIMS_FOR_DEMOGRAPHIC_REINFORCEMENT",
    "OPINION",
    "DECIDED_TO"
]

node_properties=True
relationship_properties=True
llm_transformer = LLMGraphTransformer(
    llm=llm_text,
    allowed_nodes=allowed_nodes,
    allowed_relationships=allowed_relationships,
    node_properties=node_properties,
    relationship_properties=relationship_properties,
    strict_mode=False,
    additional_instructions="Extract ONLY 10-15 most important nodes and ONLY 5-8 most important relationships between them. Write everything in Polish.",
)

graph_documents = llm_transformer.convert_to_graph_documents([Document(text)])
print([doc for doc in graph_documents])

In [ ]:
for doc_idx, graph_doc in enumerate(graph_documents, start=1):
    print(f"\n=== DOCUMENT {doc_idx} ===")
    print("NODES:")
    for i, node in enumerate(graph_doc.nodes, start=1):
        print(f"  {i:02d}. {node.id} ({node.type})")

    print("RELATIONSHIPS:")
    for i, rel in enumerate(graph_doc.relationships, start=1):
        source = rel.source.id
        target = rel.target.id
        rel_type = rel.type
        print(f"  {i:02d}. {source} ──[{rel_type}]──▶ {target}")

In [ ]:
from langchain_experimental.graph_transformers import LLMGraphTransformer

allowed_nodes = [
    "Person",
    "Country",
    "Organization",
    "Alliance",
    "Conflict",
    "Event",
    "Ideology",
    "Technology",
    "EconomicSector",
    "Culture",
    "HistoricalEvent",
    "Policy"
]
allowed_relationships = [
    "PERCEIVES_AS",
    "ASSUMES_POSSIBILITY_OF",
    "STRENGTHENS_COOPERATION",
    "SUPPLIES_TECHNOLOGY",
    "DRAWS_HISTORICAL_LESSONS",
    "DENIES_NECESSITY_OF",
    "DIVIDES",
    "FEARS_DEMOCRATIZATION",
    "BECOMES_DEPENDENT_ON",
    "AIMS_FOR_DEMOGRAPHIC_REINFORCEMENT",
    "OPINION",
    "DECIDED_TO"
]


node_properties=True
relationship_properties=True
llm_transformer = LLMGraphTransformer(
    llm=llm_text,
    allowed_nodes=allowed_nodes,
    allowed_relationships=allowed_relationships,
    node_properties=node_properties,
    relationship_properties=relationship_properties,
    strict_mode=False,
    additional_instructions="""Extract 7-8 most important nodes (real-world entities: countries, persons, organizations, events, concepts) and 10-12 most important factual relationships between them.

Critical rules (strict):
- DO NOT extract rhetorical, or trivial relationships, such as: believes, thinks, claims, says, suggests, fears.
- Relationships must describe objective actions, dependencies, or states with any significance.
- Each relationship must have a clear standalone meaning outside the sentence context (a single “X -> Y” must convey real information on its own).
- Write everything in Polish.

Importance filter:
- If a relationship does not change the understanding of the situation, omit it.
- Remove duplicates and trivial relationships."""
)

graph_documents = llm_transformer.convert_to_graph_documents([Document(text)])
print([doc for doc in graph_documents])

In [ ]:
for doc_idx, graph_doc in enumerate(graph_documents, start=1):
    print(f"\n=== DOCUMENT {doc_idx} ===")
    print("NODES:")
    for i, node in enumerate(graph_doc.nodes, start=1):
        print(f"  {i:02d}. {node.id} ({node.type})")

    print("RELATIONSHIPS:")
    for i, rel in enumerate(graph_doc.relationships, start=1):
        source = rel.source.id
        target = rel.target.id
        rel_type = rel.type
        print(f"  {i:02d}. {source} ──[{rel_type}]──▶ {target}")